# 01 — Data audit, geometry confound and split isolation

This notebook audits the grouped Defactify manifest before H1-N training. The official split had cross-split caption and perceptual-hash overlap, so the group-disjoint manifest is the internal experimental frame. The original internal test was inspected during D0; therefore any H1-N result on it is an **exploratory internal stress-test result**, not the confirmatory result.

In [ ]:
from pathlib import Path

import pandas as pd

from ai_image_detector.manifest import audit_summary, load_manifest, split_overlap_report

MANIFEST = Path('../data/processed/defactify_grouped/manifest.csv')
assert MANIFEST.exists(), 'Run prepare_defactify.py and make_grouped_split.py first.'
frame = load_manifest(MANIFEST, check_paths=True)
required_columns = {'label', 'split', 'generator', 'width', 'height', 'leakage_group'}
missing_columns = required_columns.difference(frame.columns)
assert not missing_columns, f'Manifest misses controlled-protocol columns: {sorted(missing_columns)}'
frame.head()

In [ ]:
summary = audit_summary(frame)
for name, value in summary.items():
    print(f'\n--- {name} ---')
    display(value) if hasattr(value, 'style') else print(value)

for key in ('source_id', 'group_id', 'leakage_group', 'caption', 'phash'):
    if key in frame.columns:
        leaked = split_overlap_report(frame, key)
        print(f'{key}: {len(leaked)} records in a cross-split group')
        if len(leaked):
            display(leaked.head())

## Why D0 required an amendment

The prepared corpus has a class-correlated geometry/source channel: real photographs have varied rectangular dimensions, whereas synthetic images are square, and no exact `(width, height)` pair is shared across the two labels. D0's metadata-only control and direct rectangular-to-square resizing therefore showed that geometry can produce a high score without demonstrating image provenance. In particular, anisotropic resizing can create a class-correlated frequency pattern before an FFT is calculated.

D0 remains a useful *diagnostic* record. It is not an H1-N baseline, an architecture-selection result, or evidence that an individual image is AI-generated.

In [ ]:
geometry = (
    frame.assign(aspect_ratio=frame['width'] / frame['height'])
    .groupby(['label', 'generator'], dropna=False)
    .agg(
        images=('path', 'size'),
        unique_widths=('width', 'nunique'),
        unique_heights=('height', 'nunique'),
        median_aspect_ratio=('aspect_ratio', 'median'),
    )
    .reset_index()
)

real_dimensions = set(map(tuple, frame.loc[frame.label == 0, ['width', 'height']].to_numpy()))
fake_dimensions = set(map(tuple, frame.loc[frame.label == 1, ['width', 'height']].to_numpy()))
geometry_gate = {
    'exact_width_height_pairs_shared_between_labels': len(real_dimensions & fake_dimensions),
    'real_images_square_fraction': float((frame.loc[frame.label == 0, 'width'] == frame.loc[frame.label == 0, 'height']).mean()),
    'fake_images_square_fraction': float((frame.loc[frame.label == 1, 'width'] == frame.loc[frame.label == 1, 'height']).mean()),
}
display(geometry)
geometry_gate

## H1-N paired group sampler

The neural H1-N RGB/FFT runs train from the group-disjoint manifest with `paired_group_balanced_v1`. Each group visit emits one real image and one fake sibling; each neural training image then receives its seeded random square crop. The fake generator is assigned from a balanced, seeded cycle, so large duplicate groups and a more frequent generator cannot obtain extra training weight. Neural validation/test use the deterministic centre crop. The controlled radial-logistic baseline does not use this sampler or random crop: it extracts one deterministic centre-crop feature vector per image on every split. This is a training balance mechanism; it does not make the internal test confirmatory.

In [ ]:
from itertools import islice

from ai_image_detector.training import (
    PAIRED_GROUP_BALANCED_SAMPLER,
    PairedGroupSampler,
)

train_frame = frame.loc[frame.split == 'train'].reset_index(drop=True)
sampler = PairedGroupSampler(train_frame, seed=7, group_column='leakage_group')
sampled_indices = list(islice(iter(sampler), 12))
sampled = train_frame.iloc[sampled_indices].copy()

for start in range(0, len(sampled_indices), 2):
    pair = sampled.iloc[start : start + 2]
    assert pair.label.tolist() == [0, 1]
    assert pair.leakage_group.nunique() == 1

display(sampled[['leakage_group', 'label', 'generator', 'path']])
assert sampler.metadata()['choice'] == PAIRED_GROUP_BALANCED_SAMPLER
sampler.metadata()

## Decision gate

Proceed only if all group/caption/near-duplicate checks are clean for the grouped manifest and the sampler check passes. H1-N then uses a source-normalised 128 × 128 raster for **both** representations; Notebook 02 makes that contract visible. Do not use accuracy alone, D0 metrics, or a smoke run to choose a deployment model.